In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak
from tqdm import tqdm
from datetime import datetime, timedelta
from collections import Counter
from quant_utils import filter_stock,filter_main_board,filter_main_board_no_st,add_ts_code,run_daily_update_fast
import tushare as ts
from tqdm.auto import tqdm
from pathlib import Path
from time import sleep
from datetime import date
import requests

BY_DATE_DIR = Path("data/by_date")
BY_STOCK_DIR = Path("data/by_stock")

BY_DATE_DIR.mkdir(parents=True, exist_ok=True)
BY_STOCK_DIR.mkdir(parents=True, exist_ok=True)

ts.set_token("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")
pro = ts.pro_api("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")

#YYMMDD format
today = date.today().strftime("%Y%m%d")


In [13]:
#抓取数据（每天凌晨12：00点自动化抓取）
stock_name = ak.stock_info_a_code_name()

stock_name

,code,name,board
0,000001,平安银行,深主板
1,000002,万 科Ａ,深主板
2,000004,*ST国华,深主板
3,000006,深振业Ａ,深主板
4,000007,全新好,深主板
...,...,...,...
5482,920978,开特股份,北交所
5483,920981,晶赛科技,北交所
5484,920982,锦波生物,北交所
5485,920985,海泰新能,北交所


In [14]:
stock_name

#加板块column
stock_name = filter_stock(stock_name)

#只要主板
main_board = filter_main_board(stock_name)

#加入ts_code
mainboard_with_ts_code = add_ts_code(main_board)

#过滤ST和*ST
no_ST_mainboard_with_ts_code = filter_main_board_no_st(mainboard_with_ts_code)

stock_name.to_csv("data/all_stock_name.csv", index=False, encoding="utf-8-sig")
main_board.to_csv("data/main_board.csv", index=False, encoding="utf-8-sig")
mainboard_with_ts_code.to_csv("data/main_board_with_ts_code.csv", index=False, encoding="utf-8-sig")
no_ST_mainboard_with_ts_code.to_csv("data/no_ST_mainboard_with_ts_code.csv", index=False, encoding="utf-8-sig")


In [15]:
today_df = run_daily_update_fast(mainboard_with_ts_code, trade_date=today)
print(today_df.shape)
display(today_df.head())


(3185, 15)


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor,code,name,board
0,000001.SZ,2026-03-04,10.84,10.84,10.67,10.71,10.88,-0.17,-1.5625,1089713.96,1167995.856,134.5794,000001,平安银行,深主板
1,000002.SZ,2026-03-04,4.64,4.68,4.59,4.62,4.67,-0.05,-1.0707,1176896.39,544991.410,181.7040,000002,万 科Ａ,深主板
2,000004.SZ,2026-03-04,5.92,5.92,5.92,5.92,6.23,-0.31,-4.9759,3010.00,1781.920,4.0640,000004,*ST国华,深主板
3,000006.SZ,2026-03-04,8.37,8.70,8.31,8.56,8.56,0.00,0.0000,182373.00,156255.311,39.7400,000006,深振业Ａ,深主板
4,000007.SZ,2026-03-04,12.49,13.19,12.40,13.17,12.60,0.57,4.5238,92538.51,119574.740,8.2840,000007,全新好,深主板


In [ ]:


payload = {
    "api_name": "stock_basic",
    "token": "你的token",
    "params": {},
    "fields": "ts_code,symbol,name"
}

try:
    r = requests.post("https://api.tushare.pro", json=payload, timeout=30)
    print("status:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("error:", repr(e))

status: 200
{"request_id":"","code":40101,"data":null,"msg":"您的token不对，请确认。"}


In [7]:
df = pd.read_parquet("/Users/ciciliu/stock_analysis/data/by_date/2000-01-10.parquet")
print(df.head())
df.to_csv("data/example.csv", index=False, encoding="utf-8-sig")

     ts_code trade_date   open   high    low  close  pre_close  change  \
0  000001.SZ 2000-01-10  19.79  20.48  19.77  20.14      19.54    0.60   
1  000002.SZ 2000-01-10  10.68  11.44  10.55  11.44      10.40    1.04   
2  000004.SZ 2000-01-10  10.28  10.36  10.01  10.36       9.87    0.49   
3  000006.SZ 2000-01-10  10.50  11.00  10.19  10.82      10.45    0.37   
4  000007.SZ 2000-01-10   9.50   9.99   9.45   9.61       9.33    0.28   

   pct_chg       vol       amount  adj_factor    code   name board  
0     3.07  185211.0  372294.4957      21.662  000001   平安银行   深主板  
1    10.00  142425.0  159259.0559       8.995  000002  万  科Ａ   深主板  
2     4.96   24623.0   25405.2053         NaN  000004  *ST国华   深主板  
3     3.54   27733.0   29816.0204       4.111  000006   深振业Ａ   深主板  
4     3.00   41236.0   39841.2720       3.632  000007    全新好   深主板  


In [51]:

daily_000001 = pro.daily(ts_code="000001.SZ",start_date="19910404",end_date = "19910909").sort_values("trade_date").reset_index(drop=True)

daily_000001

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,000001.SZ,19910404,48.76,48.76,48.76,48.76,49.00,-0.24,-0.49,3.0,15.0
1,000001.SZ,19910405,48.52,48.52,48.52,48.52,48.76,-0.24,-0.49,2.0,10.0
2,000001.SZ,19910408,48.04,48.04,48.04,48.04,48.52,-0.48,-0.99,2.0,10.0
3,000001.SZ,19910409,47.80,47.80,47.80,47.80,48.04,-0.24,-0.50,4.0,19.0
4,000001.SZ,19910410,47.56,47.56,47.56,47.56,47.80,-0.24,-0.50,15.0,71.0
...,...,...,...,...,...,...,...,...,...,...,...
83,000001.SZ,19910903,15.00,15.00,14.70,15.00,15.00,0.00,0.00,2874.0,4271.0
84,000001.SZ,19910904,14.80,14.85,14.70,14.85,15.00,-0.15,-1.00,1604.0,2367.0
85,000001.SZ,19910905,14.70,14.70,14.30,14.60,14.85,-0.25,-1.68,4840.0,7030.0
86,000001.SZ,19910906,14.30,14.30,13.50,13.70,14.60,-0.90,-6.16,3267.0,4506.0


In [39]:
list_date_000004 = pro.stock_basic(ts_code="000004.SZ", fields="ts_code,list_date")
print(list_date_000004)

     ts_code list_date
0  000004.SZ  19901201
